# 🛠️ ITI AI Track: Native Python Kafka Pipelines Lab

**Objective:** Build an end-to-end event-driven architecture using pure Python clients. You will manage partitions manually, handle direct file serialization into Parquet, and run high-performance SQL analytical queries directly on those files using DuckDB.

---

## 📋 Unified Data Schema Definitions

To keep things clean, all components must strictly follow these schemas:

1. **`topic_raw` / `topic_fraud` JSON Schema:**

```json
{
  "transaction_id": "str",
  "user_id": "int",
  "amount": "float",
  "location": "str"
}
```

2. **`sales_topic` Avro Schema:**

```json
{
  "order_id": "int",
  "item_name": "string",
  "price": "float"
}
```

---

## 🏗️ Environment Setup

### Step 1: Spin Up Infrastructure

Run the following command inside your GitHub Codespace terminal to launch your Docker cluster containing Kafka, Schema Registry, and MinIO:

```bash
docker compose up -d
```

### Step 2: Install Libraries

Install the native Python client libraries needed to interface with the cluster and handle file structures:

```bash
pip install confluent-kafka[avro] fastavro pandas pyarrow duckdb
```

### Step 3: Create Topics Manually

Run commands in your workspace terminal to prepare your message streams with dedicated partition properties:

---

# 🔍 Task 1: Real-Time Fraud Pipeline, Parquet Sinking & DuckDB Analytics

## Step 4: Write the Partition-Targeted Producer (`producer_advanced.py`)

This script acts as the data generator. It sends transaction data to explicit partitions based on a structural business rule.

**Logic**

- If `location == "Cairo"` → send to **Partition 0**
- If `location == "Alexandria"` → send to **Partition 1**

---

## Step 5: Write the Partition-Isolated Consumer (`consumer_partition.py`)

This script mimics an ML scoring engine.

Instead of subscribing to all partitions, it is manually pinned to **Partition 0** only.

**Logic**

- Consume Cairo transactions only.
- If `amount > 100000`, forward the record to `topic_fraud`.

---

## Step 6: Test Offset Strategies (`test_offsets.py`)

Use this script to compare offset behaviors.

### Experiment

- `latest` → ignore historical records.
- `earliest` → replay historical records.

---

## Step 7: Write the Local Parquet Lakehouse Sink (`consumer_to_parquet.py`)

This consumer accumulates fraud alerts and periodically writes them as Parquet files.

---

## Step 8: Run Serverless Analytics via DuckDB (`analytics.py`)

Query all Parquet files directly without Spark or a database server.

---

# 🏁 Lab Execution Run-Order Checklist

Open multiple terminal windows and run the components in the following order:

### Terminal 1

```bash
python consumer_to_parquet.py
```

### Terminal 2

```bash
python consumer_partition.py
```

### Terminal 3

```bash
python producer_advanced.py
```

### Terminal 4

After Parquet files appear inside `data_lake/`:

```bash
python analytics.py
```

### Terminal 4 (Next Step)

```bash
python producer_avro.py
```

---

## Expected Flow

```text
Producer
    │
    ▼
topic_raw (2 partitions)
    │
    ▼
Partition-Isolated Consumer (Partition 0 only)
    │
    ▼
topic_fraud
    │
    ▼
Parquet Sink
    │
    ▼
data_lake/*.parquet
    │
    ▼
DuckDB Analytics
```
